# Deploying AI
## Assignment 2: AI system with API Queries, and Semantic Search in a conversational interface

The goal of this assignment is to design and implement an AI system with a conversational interface.

# Load Secrets

In [5]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Data Collection & Processing Pipeline

The Uncle Joe chatbot uses a multi-stage pipeline for collecting and processing Trader Joe's product data. All scripts are located in `05_src/assignment_chat/scripts/`.

### Pipeline Overview

1. **URL Collection** → 2. **Product Scraping** → 3. **Embedding Generation** → 4. **ChromaDB Setup**

### Step 1: Product URL Collection

**Script:** `05_src/assignment_chat/scripts/scrape_product_urls.py`

**Purpose:** Collects product URLs from Trader Joe's website using Playwright for async web scraping.

**Key Features:**
- Scrapes all product listing pages starting from the main category page
- Uses Playwright with anti-detection techniques (custom user agent, stealth scripts)
- Handles pagination automatically
- Saves URLs to `data/tj_product_urls.json` with tracking metadata
- Supports headless and visible browser modes for debugging

**Output:** JSON file with product URLs and metadata (product_id, scraped status, errors)

In [ ]:
# Run URL collection script
# Note: This script uses Playwright and requires installation: playwright install chromium
# Uncomment the line below to run (takes ~5-10 minutes for full scrape)

# !cd ../05_src/assignment_chat && python scripts/scrape_product_urls.py

print("URL collection script located at: 05_src/assignment_chat/scripts/scrape_product_urls.py")
print("This script has already been run and URLs saved to: 05_src/assignment_chat/data/tj_product_urls.json")

### Step 2: Product Details Scraping

**Script:** `05_src/assignment_chat/scripts/scrape_product_details.py`

**Purpose:** Scrapes comprehensive product details from each product URL collected in Step 1.

**Key Features:**
- Extracts product name, price, category, description, tags
- **Ingredients:** Scrapes from `<li>` elements in ingredient lists
- **Allergens:** Splits into `allergens_contains` and `allergens_may_contain` arrays
- **Nutrition:** Full nutrition facts table (calories, protein, fat, carbs, vitamins, minerals)
- Resume capability: Tracks scraped URLs and can continue from interruption
- Progress tracking: Saves after each product to prevent data loss
- Error handling: Logs failures and continues with remaining products

**Output:** `data/tj_products_full.json` with complete product data

In [ ]:
# Run product details scraper
# Note: This takes a long time (~1-2 hours for ~1500 products)
# Uncomment to run:

# !cd ../05_src/assignment_chat && python scripts/scrape_product_details.py

print("Product details scraper located at: 05_src/assignment_chat/scripts/scrape_product_details.py")
print("Output saved to: 05_src/assignment_chat/data/tj_products_full.json")

### Step 3: Embeddings Generation

**Script:** `05_src/assignment_chat/scripts/create_embeddings_enhanced.py`

**Purpose:** Generates OpenAI embeddings for semantic search using product data from Step 2.

**Key Features:**
- Uses OpenAI's `text-embedding-3-small` model (1536 dimensions)
- Creates rich text representations including: name, category, description, tags, ingredients, nutrition highlights
- Processes products in batches of 50 to optimize API usage
- **Archival:** Automatically backs up existing embeddings file with timestamp before creating new one
- Stores embedding alongside original product data
- Cost-efficient: ~$0.02 per 1000 products

**Requirements:**
- `OPENAI_API_KEY` environment variable must be set
- Input: `data/tj_products_full.json` from Step 2

**Output:** `data/tj_products_with_embeddings.json` (includes embeddings + product data)

In [ ]:
# Generate embeddings
# Note: Requires OPENAI_API_KEY environment variable
# Takes ~2-5 minutes for ~1500 products, costs ~$0.02-0.03
# Uncomment to run:

# !cd ../05_src/assignment_chat && python scripts/create_embeddings_enhanced.py

print("Embeddings generation script: 05_src/assignment_chat/scripts/create_embeddings_enhanced.py")
print("Output: 05_src/assignment_chat/data/tj_products_with_embeddings.json")
print("Archives existing embeddings automatically before creating new ones")

### Step 4: ChromaDB Setup

**Script:** `05_src/assignment_chat/scripts/setup_chromadb.py`

**Purpose:** Loads pre-computed embeddings into ChromaDB for semantic search.

**Key Features:**
- Creates persistent ChromaDB collection at `chroma_db/` directory
- Uses cosine similarity for vector search
- Loads embeddings from `data/tj_products_with_embeddings.json`
- Interactive mode: Prompts before overwriting existing collection
- Stores metadata (name, price, category, URL) alongside embeddings
- Processes in batches of 100 for efficiency
- Database size: ~15-25 MB (well under 40 MB limit)

**Important Notes:**
- Uses `collection.add()` which does NOT support upsert
- If you answer 'n' to reset, re-running will cause duplicate ID errors
- To refresh data: answer 'y' to reset prompt
- Includes test search to verify setup

**Output:** ChromaDB persistent storage in `chroma_db/` directory

In [ ]:
# Setup ChromaDB
# Note: Interactive script that prompts for reset confirmation
# Uncomment to run:

# !cd ../05_src/assignment_chat && python scripts/setup_chromadb.py

print("ChromaDB setup script: 05_src/assignment_chat/scripts/setup_chromadb.py")
print("Output: 05_src/assignment_chat/chroma_db/ directory")
print("Run this after embeddings are generated")

## Uncle Joe's Services

The chatbot provides three core services, each fulfilling assignment requirements. All services are located in `05_src/assignment_chat/services/`.

### Service 1: Nutrition API Service (**Requirement #1: API Calls**)

**File:** `05_src/assignment_chat/services/nutrition_service.py`

**Purpose:** Provides nutritional information using the Open Food Facts API.

**API Used:** [Open Food Facts API](https://world.openfoodfacts.org/) - completely free, no authentication required

**Key Features:**
- Searches for products by name (searches for "trader joe {product_name}")
- Retrieves nutrition data: calories, protein, fat, carbs, fiber, sugar, sodium, vitamins, minerals
- Also fetches ingredients and allergen information
- **Transformation:** Converts structured API responses into natural language Uncle Joe's voice (not verbatim)
- Generic query detection: Detects generic queries like "high protein snacks" vs specific products
- Fallback handling: Returns friendly message when product not found

**Key Methods:**
- `analyze_product()`: Main method to get nutrition data
- `get_ingredients()`: Fetches ingredients and allergens
- `format_for_uncle_joe()`: Transforms API data into Uncle Joe's personality

**Example Transformation:**
- API returns: `{"calories": 150, "protein_g": 8, "fat_g": 4}`
- Uncle Joe says: "Per serving: 150 calorie, 8g protein, 4g fat. Good protein (8g) - help build muscle!"

### Service 2: Product Search Service (**Requirement #2: Semantic Query**)

**File:** `05_src/assignment_chat/services/product_search_service.py`

**Purpose:** Semantic search across Trader Joe's product catalog using vector embeddings.

**Vector Database:** ChromaDB with persistent file storage (not Docker)

**Key Features:**
- **Semantic search:** Uses OpenAI `text-embedding-3-small` embeddings (1536 dimensions)
- **Hybrid search:** Combines vector similarity with metadata filtering (price, category)
- Queries ChromaDB using cosine similarity
- Fallback to metadata-based search when API key unavailable
- Returns top N results with similarity scores
- Supports filter options: max_price, min_price, category, categories

**Key Methods:**
- `search()`: Main semantic search with optional filters
- `get_ingredients()`: Retrieve ingredients for a product from local DB
- `format_for_uncle_joe()`: Format search results in Uncle Joe's voice
- `_build_where_filter()`: Construct ChromaDB metadata filters

**Data Source:**
- Dataset: ~1500 Trader Joe's products
- Storage: ChromaDB persistent client (file-based, ~20 MB)
- Enrichment: Includes name, price, category, description, tags, ingredients, nutrition

**Example Query:**
- User: "Show me plant-based snacks under $5"
- System: Generates embedding → Queries ChromaDB with price filter → Returns top 5 matches

### Service 3: Recipe Service (**Requirement #3: Function Calling**)

**File:** `05_src/assignment_chat/services/recipe_service.py`

**Purpose:** Generates recipe ideas using GPT-4o-mini with function calling to search products.

**Tool Used:** Function Calling (Tool choice from assignment requirements)

**Key Features:**
- Uses GPT-4o-mini model with function calling capability
- **Function:** `search_products_for_recipe` - searches ChromaDB for products by name/category
- Recipe generation flow:
  1. User requests recipe (e.g., "give me a pasta recipe")
  2. LLM decides which products to search for
  3. Calls `search_products_for_recipe` function
  4. Receives actual Trader Joe's products
  5. Generates recipe using those specific products
- Returns recipes in Uncle Joe's voice with step-by-step instructions
- Focuses on Trader Joe's products only (no generic ingredients)

**Function Calling Example:**
```python
tools = [{
    "type": "function",
    "function": {
        "name": "search_products_for_recipe",
        "description": "Search for Trader Joe's products to use in recipe",
        "parameters": {...}
    }
}]
```

**Why Function Calling:**
- Allows LLM to autonomously decide when to search products
- More dynamic than hard-coded product lookups
- LLM can search multiple times with different queries to gather ingredients

## Uncle Joe's Chat Application

**File:** `05_src/assignment_chat/uncle_joe_app.py`

**Purpose:** Main Gradio-based chatbot interface that ties all services together.

### Key Features

**Personality: Uncle Joe**
- Malaysian uncle character inspired by Uncle Roger
- Uses expressions like "Haiyaa" (disappointment), "Fuiyoh" (excitement)
- Friendly, enthusiastic, sometimes critical but always helpful
- Maintains conversational memory throughout session

**Intent Detection (LLM-based)**
- Uses GPT-4o-mini to classify user queries into intents:
  - `nutrition`: Nutritional information requests
  - `ingredients`: Ingredient/allergen queries
  - `search`: Product search queries
  - `recipe`: Recipe suggestions
  - `chat`: General conversation
  - `off_topic`: Restricted topics (cats, dogs, horoscopes, Taylor Swift)
- Fallback to rule-based detection if LLM fails

**Intent Handlers**
1. **Nutrition Handler:** Calls NutritionService → Open Food Facts API
2. **Ingredients Handler:** Dual-source approach (local ChromaDB + Open Food Facts)
3. **Search Handler:** Calls ProductSearchService → ChromaDB semantic search
4. **Recipe Handler:** Calls RecipeService → GPT-4o-mini with function calling
5. **Chat Handler:** Generic GPT-4o-mini conversation in Uncle Joe's voice
6. **Off-topic Handler:** Politely refuses restricted topics with humor

**Guardrails**
- Prevents access to system prompt
- Blocks prompt modification attempts
- Refuses restricted topics (cats, dogs, horoscopes, Taylor Swift)
- System prompt instructs Uncle Joe to deflect these topics naturally

**Memory Management**
- Maintains conversation history in Gradio's `state`
- Currently unlimited (can be extended with context window management)

**UI: Gradio Interface**
- Port: 7860
- Chatbot component with message history
- Text input with submit button
- Clean, simple interface

In [ ]:
# Run Uncle Joe's Chat Application
# Note: Requires OPENAI_API_KEY and ChromaDB to be set up first
# Launches Gradio interface on http://localhost:7860
# Uncomment to run:

# !cd ../05_src/assignment_chat && python uncle_joe_app.py

print("Uncle Joe's Chat App: 05_src/assignment_chat/uncle_joe_app.py")
print("Run with: cd 05_src/assignment_chat && python uncle_joe_app.py")
print("Access at: http://localhost:7860")

# Requirements

Your project should meet the following specifications.

## Services

You must include at least **three services** in your system.

### Service 1: API Calls

* One service must use an API as its back end.
* You can refer to the list of [public and free APIs on GitHub](https://github.com/public-apis/public-apis).
* This service may simply return the API’s output to the user, but the response must not be provided verbatim. Instead, transform or rephrase the output, for example, by summarizing, rewriting in a natural tone, or converting structured data into written text.

### Service 2: Semantic Query

* One service must allow users to ask questions that are resolved through a semantic search (or a hybrid approach, such as lexical search followed by semantic search).
* You may use the datasets introduced in class, or choose your own dataset. 

If you use your own dataset:
* Please **limit file sizes to 40 MB**, so it can be easily shared via GitHub. Note that GitHub warns about files over 50 MB and we generally want to avoid uploading large files.
* **Do not expect us to run the code use to produce embeddings** in the repository. You can include the code used to produce the embeddings, but we ask you to describe your embedding process in the project’s README file.
* Use a [ChromaDB instance with file persistence](https://docs.trychroma.com/docs/run-chroma/persistent-client). This is similar to the first implementation used in class but smaller and easier to host than the Docker-based version.
* If your app needs to access structured data (e.g., to enrich query results), you may use CSV files read with pandas as a back end.
* Please do not use SQLite. We did not include a SQLite library in your environment.

### Service 3: Your Choice

* The third service is open-ended: you may design it as you wish.
* It must make use of one of the following tools:

  * [Function Calling](https://platform.openai.com/docs/guides/function-calling) (API calling is acceptable, but not mandatory)
  * [Web Search](https://platform.openai.com/docs/guides/tools-web-search?api-mode=responses): You may perform simple web searches; if you use **agentic searches**, justify your decision. Avoid using “Deep Research.”
  * [MCP Server Connection](https://platform.openai.com/docs/guides/tools-connectors-mcp): You can explore available servers on [glama.ai](https://glama.ai/mcp/servers).

## User Interface

* The system must include a chat-based interface, preferably implemented with Gradio.
* Give the chat client a distinct personality to make the interaction engaging. For example, assign a specific tone, role, or conversational style.
* The chat interface must maintain memory throughout the conversation.

  * (Optional) Implement a memory management system for long conversations. You don’t need long-term memory, but you should demonstrate how your system handles situations when a conversation becomes too long for the context window.
  * (Optional) You may decide the context window’s size, but remember that full coverage of the entire conversation is not required. A useful reference is ['Manage short-term memory' from LangGraph](https://docs.langchain.com/oss/python/langgraph/add-memory#manage-short-term-memory).

---

## Guardrails and Other Limitations

* Include guardrails that prevent users from:

  * Accessing or revealing the system prompt.
  * Modifying the system prompt directly.

* The model must not respond to questions on certain restricted topics:

  * Cats or dogs
  * Horoscopes or Zodiac Signs
  * Taylor Swift

## Implementation

+ Implement your code in the folder `./05_src/assignment_chat`.
+ Add a `readme.md` where you explain the nature of your chat client, the serivices that it provides, and any decisions that you made related to the implementation.
+ We will not be able to install more libraries to assess your work. Please use the standard setup of the course.

# Submission Information

**Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/deploying-ai/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
